# Agent 구성

- 올림픽 정보 제공: Qdrant DB와 연동하는 **임베딩 벡터 Retrieve Tool**
- 영화 정보 제공: Neo4j 연동 **GraphRAG Tool**


In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

from qdrant_client import QdrantClient

from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
################################################
# QDrant olympic-wiki Collectio 연동 tool
################################################
from langchain_core.tools.retriever import create_retriever_tool

COLLECTION_NAME = "olympic_info_wiki"
VECTOR_SIZE = 3072

client = QdrantClient(url="http://localhost:6333")

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = QdrantVectorStore(
    client=client,
    embedding=embeddings, 
    collection_name=COLLECTION_NAME 
    # ,vector_name="dense"
)

# VectorStore 를 Retriever로 변환
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 7}
)

# Tool로 변환 
search_olympic_info_tool = create_retriever_tool(
    retriever=retriever,
    name="search_olympic_info",
    description="VectorDB의 Olympic 정보를 유사도 검색을 통해 조회하는 도구"
)

In [ ]:
#####################################################################
# 영화 정보를 제공하는 GraphDB 연동 RAG tool
#####################################################################
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain
from langchain_neo4j import Neo4jGraph
from langchain_neo4j.chains.graph_qa.cypher import CYPHER_GENERATION_PROMPT

# cypher search 쿼리를 실행하지 못한다.
custom_prompt_str = CYPHER_GENERATION_PROMPT.template+\
"""
<constraint>
- 검색조건이 영화제목, 감독이나 배우이름일 경우 입력값을 영어로 변경해서 검색한다. (예: "영화 '인셉션'의 감독과 개봉일을 알려줘." ->" 영화 'Inception'의 감독과 개봉일을 알려줘.")
- Cypher 쿼리는 NEO4J 5.0 문법에 맞게 작성한다.
</constraint>
"""

graph = Neo4jGraph()
prompt_template = PromptTemplate(template=custom_prompt_str)
cypher_chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(model="gpt-5.5", temperature=0.0), 
    graph=graph, 
    allow_dangerous_requests=True,
    verbose=True,
    cypher_prompt=prompt_template
)

In [ ]:
query = "크리스토퍼 놀란의 영화에 가장 많이 출연한 배우는 누구인가?"
cypher_chain.invoke({"query": query})

In [ ]:
#################################
# TOOL 로 변환
#################################
@tool
def search_movie_info_tool(query: str) -> str:
    """NEO4J 에서 영화 정보를 검색하는 데 사용하는 도구입니다. 그래프 데이터베이스에 저장된 엔티티 간 관계, 구조적 질의(예: 'A와 연결된 B는?', '몇 개의 관계가 있는가')에 답할 때 사용하세요"""
    response = cypher_chain.invoke({"query": query})
    return response["result"]


# search_movie_info.invoke("크리스토퍼 놀란의 영화에 가장 많이 출연한 배우는 누구인가?")

In [ ]:
######################################################
# Agent 생성 
# llm + search_movie_info, search_olympic_info_tool 
######################################################
from langchain.agents import create_agent

agent = create_agent(
    model=ChatOpenAI(model="gpt-5.5"),
    tools=[search_movie_info_tool, search_olympic_info_tool],
    system_prompt="""올림픽와 영화정보를 친절히 설명하는 유능한 Agent입니다. 
영화정보 검색을 위한 search_movie_info, 올림픽관련 정보를 검색하기 위한 search_olympic_info_tool 두개의 tool을 사용할 수 있습니다."""
)

In [ ]:
# query = "톰 크루즈와 같은 영화에 출연한 배우들은 누가 있나?"
# query = "런던 올림픽이 개최된 년도와 그때 개봉한 영화는 뭐가 있나?"
query = "2차 세계대전을 배경으로 한 영화들을 추천해줘."

res = agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
)

In [ ]:
res['messages']

In [ ]:
print(res['messages'][-1].content)